
# 用 `T_SR + T_GR_MeTp` 直接计算 `T_MeTp`

本 notebook 按修正后的关系直接计算：

\[
T_{\mathrm{MeTp}} = T_{\mathrm{SR,MeTp}} + T_{\mathrm{GR,MeTp}}
\]

其中：

- `T_SR.csv` 中的 `T_SR` 列作为 `T_SR,MeTp`
- `LightTime_T_GR_MeTp.csv` 中的 `delta_t_s` 列作为 `T_GR,MeTp`

按 `gps_time` 对齐后直接相加得到 `T_MeTp`。


In [11]:

import numpy as np
import pandas as pd
import csv

LD = np.longdouble

TSR_PATH = "T_SR.xlsx"
GR_PATH = "LightTime_T_GR_MeTp.xlsx"

OUT_XLSX = "T_MeTp.xlsx"


def to_longdouble_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().map(np.longdouble)


def read_tsr_xlsx(filepath: str) -> pd.DataFrame:
    df = pd.read_excel(filepath, dtype=str)
    if "gps_time" not in df.columns or "T_SR" not in df.columns:
        raise KeyError(f"{filepath} 中必须包含 gps_time 和 T_SR 列")

    out = df[["gps_time", "T_SR"]].copy()
    out["gps_time"] = out["gps_time"].astype(np.int64)
    out["T_SR_raw"] = out["T_SR"].astype(str).str.strip()
    out["T_SR_num"] = to_longdouble_series(out["T_SR"])
    return out.drop(columns=["T_SR"]).sort_values("gps_time").reset_index(drop=True)


def read_gr_metp_xlsx(filepath: str) -> pd.DataFrame:
    df = pd.read_excel(filepath, dtype=str)
    if "gps_time" not in df.columns or "delta_t_s" not in df.columns:
        raise KeyError(f"{filepath} 中必须包含 gps_time 和 delta_t_s 列")

    out = df[["gps_time", "delta_t_s"]].copy()
    out["gps_time"] = out["gps_time"].astype(np.int64)
    out["T_GR_MeTp_raw"] = out["delta_t_s"].astype(str).str.strip()
    out["T_GR_MeTp_num"] = to_longdouble_series(out["delta_t_s"])
    return out.drop(columns=["delta_t_s"]).sort_values("gps_time").reset_index(drop=True)


def stringify_longdouble(x):
    if pd.isna(x):
        return ""
    if isinstance(x, (np.longdouble, np.float64, float, np.floating)):
        return np.format_float_scientific(np.longdouble(x), precision=18, unique=False, trim='k')
    return str(x)


In [12]:

tsr = read_tsr_xlsx(TSR_PATH)
gr = read_gr_metp_xlsx(GR_PATH)

df = (
    tsr.merge(gr, on="gps_time", how="inner")
       .sort_values("gps_time")
       .reset_index(drop=True)
)

df["T_MeTp_num"] = df["T_SR_num"].astype(object) + df["T_GR_MeTp_num"].astype(object)

print("rows =", len(df))
print("gps_time range:", int(df["gps_time"].iloc[0]), "->", int(df["gps_time"].iloc[-1]))
df.head()


rows = 86400
gps_time range: 707659200 -> 707745599


,gps_time,T_SR_raw,T_SR_num,T_GR_MeTp_raw,T_GR_MeTp_num,T_MeTp_num
0,707659200,1.65613995376177e-08,1.656140e-08,8.421255814638719e-13,8.421256e-13,0.0
1,707659201,1.65613998427874e-08,1.656140e-08,8.421282704599671e-13,8.421283e-13,0.0
2,707659202,1.65614000361285e-08,1.656140e-08,8.421309541064928e-13,8.421310e-13,0.0
3,707659203,1.6561400117729e-08,1.656140e-08,8.421336323943268e-13,8.421336e-13,0.0
4,707659204,1.65614000876698e-08,1.656140e-08,8.42136305314269e-13,8.421363e-13,0.0


In [13]:

# 完整结果表（保留原始两列文本 + 新计算结果）
result = pd.DataFrame({
    "gps_time": df["gps_time"].astype(str),
    "T_SR": df["T_SR_raw"].astype(str),
    "T_GR_MeTp": df["T_GR_MeTp_raw"].astype(str),
    "T_MeTp": df["T_MeTp_num"].map(stringify_longdouble),
})

# 仅保留 gps_time 和 T_MeTp 的精简结果
result_simple = result[["gps_time", "T_MeTp"]].copy()

result.head()


,gps_time,T_SR,T_GR_MeTp,T_MeTp
0,707659200,1.65613995376177e-08,8.421255814638719e-13,1.656224166319916365e-08
1,707659201,1.65613998427874e-08,8.421282704599671e-13,1.656224197105786098e-08
2,707659202,1.65614000361285e-08,8.421309541064928e-13,1.656224216708260639e-08
3,707659203,1.6561400117729e-08,8.421336323943268e-13,1.656224225136139533e-08
4,707659204,1.65614000876698e-08,8.42136305314269e-13,1.656224222397511284e-08


In [14]:

# 写 xlsx

result.to_excel(OUT_XLSX, index=False)


print(f"saved xlsx to {OUT_XLSX}")
print(result_simple.head())


saved xlsx to T_MeTp.xlsx
    gps_time                    T_MeTp
0  707659200  1.656224166319916365e-08
1  707659201  1.656224197105786098e-08
2  707659202  1.656224216708260639e-08
3  707659203  1.656224225136139533e-08
4  707659204  1.656224222397511284e-08


In [ ]:
print('本版本按修正后的关系直接计算：T_MeTp = T_SR + T_GR_MeTp，其中 T_SR 取自第一个表格的 T_SR 列，T_GR_MeTp 取自第二个表格的 delta_t_s 列。')